# 02 · Exploración rápida con Plotly Express
**En una frase:** probamos muchos gráficos rápidos para encontrar la historia y nos quedamos solo con los tres que la sostienen.

**Detalle técnico:** corresponde al paso 4 de la guía. Usamos la metáfora de Nussbaumer Knaflic (2015): para encontrar unas pocas **perlas** (los gráficos que pasan al artefacto final) hay que abrir muchas **ostras** (gráficos exploratorios que luego se descartan).

> Estos gráficos son **exploratorios**: son para el equipo, no para la audiencia, así que usan los colores y leyendas por defecto. Después de cada uno anotamos qué aprendimos y si pasa al artefacto final (✅ perla) o se descarta (🦪 ostra).

**Regla de comparación:** los datos de 2026 llegan hasta septiembre, así que todos los años se comparan solo con los meses de **enero a septiembre**. De lo contrario, los años completos parecerían tener más casos solo por tener más meses.

In [1]:
# Configuración del entorno: el notebook vive en /notebooks, el código en /src
import sys, pathlib
RAIZ = pathlib.Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\ADRIAN\Desktop\Ciencia de datos\Visualizacion de datos\Proyecto_Plotly_SDSCJ


In [2]:
import plotly.express as px
import plotly.io as pio
from src import normalizacion as N, analisis as A, config as C

pio.templates.default = "plotly_white"
df = N.cargar()
dfv, UA, UM = A.ventana_comparable(df)
print(f"Datos: {df['fecha'].min():%Y-%m} a {df['fecha'].max():%Y-%m} · ventana comparable: meses 1–{UM}")

Datos: 2018-01 a 2026-09 · ventana comparable: meses 1–9


## 2.1 ¿Qué delitos pesan más? (volumen total en la ventana comparable)

In [3]:
anual = A.tabla_anual(dfv)
largo = anual.reset_index().melt(id_vars="delito", var_name="anio", value_name="casos")
px.bar(largo, x="anio", y="casos", color="delito", title="2.1 · Casos por delito y año (ene–sep)")

**Lectura:** el hurto a personas concentra cerca de la mitad de los casos y opaca visualmente al resto. La violencia intrafamiliar es el segundo delito y su segmento crece año a año. Con barras apiladas no se comparan bien los segmentos superiores, porque no comparten una línea base.
→ 🦪 **Ostra**, aunque nos da la primera pista.

## 2.2 Evolución mensual de todos los delitos (el espagueti)

In [4]:
mensual = A.serie_mensual(df, df["delito"].unique().tolist())
px.line(mensual, x="fecha", y="casos", color="delito", title="2.2 · Casos mensuales por delito, 2018–2026")

**Lectura:** se ven la caída de abril de 2020 (confinamiento) y el pico de hurto en 2022–2023. Pero con 12 líneas en escalas tan distintas no se distinguen las tendencias de los delitos pequeños. Además, los **últimos meses publicados** caen de golpe (p. ej., diciembre de 2025 y septiembre de 2026): es rezago de registro, no un descenso real.
→ 🦪 **Ostra**, con una alerta ética: no terminar ninguna serie en un mes incompleto.

## 2.3 Poner todos los delitos en la misma escala: índice (promedio 2018–2022 = 100)

In [5]:
indice = A.tabla_indice(anual)
idx = indice.reset_index().melt(id_vars="delito", var_name="anio", value_name="indice")
fig = px.line(idx, x="anio", y="indice", color="delito", markers=True,
              title="2.3 · Índice de denuncias ene–sep (2018–2022 = 100)")
fig.add_hline(y=100, line_dash="dot")
fig

In [6]:
resumen = A.resumen_delitos(anual, UA)
resumen

,base_2018_2022,casos_2026,cambio_abs,cambio_pct,anios_consecutivos_al_alza,participacion_pct
Violencia intrafamiliar,26767.6,43013,16245.4,60.7,3,24.8
Hurto a personas,80089.4,86376,6286.6,7.8,0,49.8
Delitos sexuales,4460.2,6613,2152.8,48.3,0,3.8
Lesiones personales,16544.8,17636,1091.2,6.6,2,10.2
Extorsión,842.2,1639,796.8,94.6,0,0.9
Hurto de motocicletas,3231.6,3392,160.4,5.0,0,2.0
Homicidios,776.2,819,42.8,5.5,0,0.5
Secuestro,7.6,23,15.4,202.6,0,0.0
Hurto a entidades financieras,31.0,4,-27.0,-87.1,0,0.0
Hurto de automotores,2668.0,2297,-371.0,-13.9,0,1.3


**Lectura (el hallazgo):** cuando cada delito se compara con su propio promedio, la **violencia intrafamiliar** es el delito que más casos suma sobre su línea base (+16.245, +61 %). También sube tres años seguidos (2023 → 2026), pero ese tramo arranca en 2023, un año con cambios de registro; por eso el titular se apoya en la comparación con el promedio 2018–2022 (y frente a 2022 el aumento es de +67 %). El hurto a personas, en cambio, baja desde su pico de 2023. La extorsión y los delitos sexuales también están por encima de su base, pero suben y bajan de un año a otro.

*Decisión ética:* la base es el promedio 2018–2022 y **no** 2023. Ese año es el más bajo de la serie y tiene muchos registros «no reportado»; usarlo como punto de partida exageraría el crecimiento.
→ ✅ **Perla 1** (Acto 1).

## 2.4 ¿Quiénes son las víctimas? Sexo

In [7]:
vif = dfv[dfv["delito"] == C.DELITO_FOCO]
sexo = vif.groupby(["anio", "sexo"], as_index=False)["casos"].sum()
px.bar(sexo, x="anio", y="casos", color="sexo", barmode="group", title="2.4 · Violencia intrafamiliar por sexo de la víctima")

**Lectura:** cerca de 7 de cada 10 víctimas son mujeres. En 2023 la proporción de hombres es atípica, otra señal de que ese año cambió la forma de registrar. → Se usa como **cifra clave (KPI)** en el encabezado, no como gráfico.

## 2.5 ¿Quiénes son las víctimas? Ciclo vital

In [8]:
ciclo = vif.groupby(["anio", "ciclo_vital"], as_index=False, observed=False)["casos"].sum()
px.bar(ciclo, x="anio", y="casos", color="ciclo_vital", title="2.5 · Violencia intrafamiliar por ciclo vital")

**Lectura:** en 2026 casi 1 de cada 4 víctimas es niña, niño o adolescente. En 2023 la categoría «No reportado» es muy alta, un problema de calidad del dato. → Se usa como **cifra clave (KPI)** en el encabezado.

## 2.6 ¿Dónde crece? Localidades

In [9]:
loc = A.vif_por_localidad(dfv, UA).reset_index()
px.bar(loc.sort_values("aumento"), x="aumento", y="localidad", orientation="h",
       title="2.6 · Aumento de casos de VIF vs. promedio 2018–2022, por localidad")

**Lectura:** el crecimiento es generalizado (18 de 20 localidades suben), pero cinco localidades explican casi la mitad del aumento absoluto. «Sin localización» se dispara por un cambio en el registro, así que se excluye del ranking y se explica en una nota.

*Limitación:* la fuente no trae población por localidad, así que no podemos calcular tasas; cada localidad se compara consigo misma.
→ ✅ **Perla 2** (Acto 2).

## 2.7 ¿Cuándo ocurre? Franja del día

In [10]:
franja = A.vif_franja_localidad(dfv, UA)
px.imshow(franja, text_auto=True, aspect="auto", color_continuous_scale="Oranges",
          title="2.7 · VIF por localidad y franja del día (2026)")

**Lectura:** la **madrugada** es la franja con más casos en 18 de las 19 localidades con casos ubicables (la excepción es Fontibón, donde lidera la tarde), y **noche + madrugada** reúnen el 61 % de los casos de 2026. Al compararlo con 2018–2022 aparece un matiz: la madrugada pierde peso (de 44 % a 36 %) porque el crecimiento se dio sobre todo en la **tarde (+143 %)** y la **noche (+96 %)**. Es un dato accionable: indica en qué horario reforzar la atención. → ✅ **Perla 3** (Acto 3), y el cambio por franja pasa a la figura de síntesis.

## 2.8 Arma o medio empleado

In [11]:
arma = vif.groupby("arma_grupo", as_index=False)["casos"].sum().sort_values("casos")
px.bar(arma, x="casos", y="arma_grupo", orientation="h", title="2.8 · VIF por arma o medio (2018–2026)")

**Lectura:** casi todos los casos ocurren **sin armas**, lo que refuerza que es una violencia cotidiana. Se menciona en el texto del Acto 3, pero un gráfico con una sola barra dominante no aporta información nueva. → 🦪 **Ostra**.

## 2.9 ¿Hay estacionalidad?

In [12]:
vif_m = df[df["delito"] == C.DELITO_FOCO].pivot_table(index="anio", columns="mes", values="casos", aggfunc="sum")
px.imshow(vif_m, text_auto=True, aspect="auto", color_continuous_scale="Oranges",
          title="2.9 · VIF por año y mes")

**Lectura:** el aumento es sostenido mes a mes desde mediados de 2023; no es un pico aislado. El último mes (septiembre de 2026) está claramente subregistrado. → 🦪 **Ostra**, pero confirma que el hallazgo es robusto.

---
## Síntesis: de la exploración a la historia (paso 3 de la guía)
**En una frase:** la exploración nos dice a quién le hablamos, qué necesita saber y qué queremos que haga.

| Elemento | Definición |
|---|---|
| **Audiencia** | Consejo Distrital de Seguridad y secretarías de Seguridad y de la Mujer de Bogotá: personas que deciden cómo asignar recursos, sin ser especialistas en datos. |
| **Qué deben saber** | Que la violencia intrafamiliar es el delito de alto impacto que más casos suma sobre su promedio (+61 %) y que el aumento se concentra en cinco localidades y en la noche y la madrugada. |
| **Qué deben hacer** | Priorizar esas localidades, extender la atención a la noche y la madrugada, y medir la VIF como indicador de seguridad. |
| **Conflicto** | La agenda pública celebra la baja del hurto mientras crece una violencia que ocurre dentro de la casa. |
| **Gran Idea** | *La violencia intrafamiliar está 61 % por encima de su promedio histórico mientras el hurto retrocede: la seguridad de Bogotá también se juega dentro de la casa.* |

**Perlas seleccionadas:** 2.3 (índice) → Acto 1 · 2.6 (localidades) → Acto 2 · 2.7 (franjas) → Acto 3 · y una figura de síntesis que reúne el cambio de las tres.
Siguiente paso: **`03_artefacto_final.ipynb`**.